# Notebook de Análise WESQUAD/DadosCandlesBacktest/2024_26

Este notebook implementa as seções solicitadas para carregar dados, gerar áreas retangulares, calcular zonas de gatilho e métricas de win/loss no universo de candles 2024_26.


## 1. Carregar bibliotecas e dados

Importar pandas/numpy e carregar os CSVs do conjunto 2024_26 para DataFrames. Se algum arquivo estiver ausente, gerar erro.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm

base_path = Path('e:/repo/PriceAction_Fisica/WESQUAD/DadosCandlesBacktest/2024_26')

symbol = 'WINJ26_F_0'
files = {tf: base_path / f'{symbol}_{tf}.csv' for tf in ['Semanal','Diário','60min','30min','15min','5min','1min']}

for k,v in files.items():
    if not v.exists():
        raise FileNotFoundError(f'Arquivo ausente para {k}: {v}')

print('Arquivos encontrados para análise:')
for k,v in files.items():
    print(k, v.name)


ModuleNotFoundError: No module named 'pandas'

## 2. Pré-processamento e normalização dos candles

Converter timestamps para datetime, ordenar por tempo, preencher misses e garantir campos opens/closes/highs/lows/volume.

In [ ]:
def load_tf(path):
    df = pd.read_csv(path, sep=';', decimal=',', encoding='latin1')
    df.columns = df.columns.str.strip()

    if 'Data' in df.columns and 'Hora' in df.columns:
        df['datetime'] = pd.to_datetime(df['Data'].astype(str) + ' ' + df['Hora'].astype(str), dayfirst=True, errors='coerce')
    elif 'datetime' in df.columns:
        df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
    elif 'Data' in df.columns:
        df['datetime'] = pd.to_datetime(df['Data'].astype(str), dayfirst=True, errors='coerce')
    else:
        raise ValueError(f"Formato inválido em {path.name}: coluna Data/Hora/datetime não encontrada")

    for col in ['Abertura', 'Máximo', 'Mínimo', 'Fechamento', 'Volume', 'Quantidade']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col].astype(str).str.replace('.', '', regex=False).str.replace(',', '.', regex=False), errors='coerce')

    required = ['datetime', 'Abertura', 'Máximo', 'Mínimo', 'Fechamento']
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Colunas obrigatórias ausentes em {path.name}: {missing}")

    df = df.dropna(subset=required)
    return df.sort_values('datetime').reset_index(drop=True)

print('Iniciando carregamento completo com tqdm...')
data = {}
for tf, path in tqdm(files.items(), desc='Carregando timeframes'):
    data[tf] = load_tf(path)
    print(f'  carregado {tf}: {len(data[tf])} candles')

print('Resumo final de carregamento:')
for tf,df in data.items():
    print(tf, '->', len(df), 'candles,', 'datetime range', df['datetime'].min(), 'to', df['datetime'].max())

## 3. Geração de áreas retangulares (high/low + x_pontos)

Parametrizar x_pontos, sl_minimo, sg_minimo, timeframe_step. Criar área de máxima e mínima com 3 candles à frente.

In [ ]:
x_pontos = 100
pt_val = 0.01
x_val = x_pontos * pt_val
sl_minimo = 50 * pt_val
sg_minimo = 80 * pt_val

# Para simplicidade, usamos timeframe_step em 15m para 60m etc; ajuste se necessário

def build_areas(df, tf_name):
    areas = []
    delta = pd.Timedelta('45m') if tf_name == '15min' else pd.Timedelta('60m')
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f'Criando áreas {tf_name}'):
        t0 = row['datetime']
        areas.append({'datetime': t0, 'type': 'max', 'y0': row['Máximo'], 'y1': row['Máximo'] + x_val, 'x0': t0, 'x1': t0 + delta})
        areas.append({'datetime': t0, 'type': 'min', 'y0': row['Mínimo'] - x_val, 'y1': row['Mínimo'], 'x0': t0, 'x1': t0 + delta})
    return pd.DataFrame(areas)

areas_15 = build_areas(data['15min'], '15min')
areas_60 = build_areas(data['60min'], '60min')

print('Áreas geradas: 15min=', len(areas_15), '60min=', len(areas_60))

print('Exemplo áreas 15min:')
print(areas_15.head())

## 4. Cálculo de toques, rejeições e métricas iniciais

Para cada área, contar toques, rejeições, direção predominante e duração. Gerar colunas métricas e JSON inicial.

In [ ]:
def area_metrics(area_df, candle_df):
    rows=[]
    for _, area in tqdm(area_df.iterrows(), total=len(area_df), desc=f'Calculando métricas área ({len(area_df)} áreas)'):
        candles = candle_df[(candle_df.datetime >= area.x0) & (candle_df.datetime <= area.x1)]
        if candles.empty:
            continue
        touches = ((candles['Máximo'] >= area.y0) & (candles['Mínimo'] <= area.y1)).sum()
        rejections = ((candles['Fechamento'] < area.y0) | (candles['Fechamento'] > area.y1)).sum()
        direction = 'up' if candles['Fechamento'].iloc[-1] >= candles['Abertura'].iloc[0] else 'down'
        rows.append({
            'datetime': area.datetime,
            'type': area.type,
            'touches': int(touches),
            'rejections': int(rejections),
            'direction': direction,
            'duration_mins': (area.x1-area.x0)/pd.Timedelta('1m')
        })
    return pd.DataFrame(rows)

metrics_15 = area_metrics(areas_15, data['15min'])
print('Area metrics sample (15min):')

print(metrics_15.head())

print('Totals: touches=', metrics_15['touches'].sum(), 'rejections=', metrics_15['rejections'].sum())

## 5. Análise multi-timeframe (Semanal/Diário/60m + 30m/15m/5m/1m)

Mapear área de contexto (Semanal/Diário/60m) e validar confluência em timeframes menores.

In [ ]:
def context_confluence(context_df, oper_df):
    out=[]
    for _,context in tqdm(context_df.iterrows(), total=len(context_df), desc=f'Confluência contexto ({len(context_df)} áreas)'):
        subset = oper_df[(oper_df.datetime >= context.datetime) & (oper_df.datetime <= context.datetime + pd.Timedelta('1h'))]
        if subset.empty:
            continue
        overlap = (subset['Máximo'].max() >= context.y0) and (subset['Mínimo'].min() <= context.y1)
        out.append({'context_time': context.datetime, 'type': context.type, 'confluent': overlap, 'n_candles': len(subset)})
    return pd.DataFrame(out)

confluence_60_15 = context_confluence(areas_60, data['15min'])
print('Confluence 60->15 count:', len(confluence_60_15), 'confluent:', confluence_60_15['confluent'].sum())

print(confluence_60_15.head())
confluence_60_15.head()


## 6. Detectar zonas de gatilho por sobreposição de 3 candles

Para cada timeframe operacional (60m/30m/15m), extrair últimos 3 candles e calcular overlap, density_score e strength.

In [ ]:
def trigger_zones(oper_df):
    zones=[]
    for i in tqdm(range(2, len(oper_df)), desc=f'Triggers zonas ({len(oper_df)} candles)'):
        c3 = oper_df.iloc[i-2:i+1]
        low_max = c3['Mínimo'].max()
        high_min = c3['Máximo'].min()
        overlap = max(0, high_min - low_max)
        rng = c3['Máximo'].max() - c3['Mínimo'].min()
        density_score = (overlap / rng * 100) if rng > 0 else 0
        strength = 'alta' if density_score >= 60 else 'média' if density_score >= 30 else 'fraca'
        zones.append({'datetime': c3.iloc[-1]['datetime'], 'price_range': [low_max, high_min], 'density_score': density_score, 'strength': strength})
    return pd.DataFrame(zones)

zones_60 = trigger_zones(data['60min'])
zones_30 = trigger_zones(data['30min'])
zones_15 = trigger_zones(data['15min'])
print('Trigger zones 15min geradas:', len(zones_15))
if not zones_15.empty:
    print(zones_15.sort_values('density_score', ascending=False).head(5))

zones_15.head()


## 7. Força de candle em 15m/5m/2m e ajuste de density_score

Calcular fForca e marcar candles com |fForca|>=55. Ajustar density_score e strength da zona.

In [ ]:
def calc_forca(df):
    frange = (df['Máximo'] - df['Mínimo']).replace(0, 0.0001)
    volmean = df['Volume'].rolling(20, min_periods=1).mean().clip(lower=1)
    fforca = ((df['Fechamento'] - df['Abertura']) / frange) * (df['Volume'] / volmean) * 100
    return fforca.clip(-100, 100)

for tf in ['15min','5min','1min']:
    data[tf]['fForca'] = calc_forca(data[tf])
    data[tf]['forca_sinal'] = data[tf]['fForca'].abs() >= 55
    print(f'Forca calc para {tf}: quantos sinais fortes =', int(data[tf]['forca_sinal'].sum()), 'total candles =', len(data[tf]))

# Ajuste simples de density score nas zonas 15min usando contagem de fForca alta nos últimos 15m
for idx,row in zones_15.iterrows():
    look = data['15min'][(data['15min'].datetime <= row['datetime']) & (data['15min'].datetime > row['datetime']-pd.Timedelta('15m'))]
    if not look.empty:
        count = look['forca_sinal'].sum()
        zones_15.at[idx,'forca_count'] = int(count)
        zones_15.at[idx,'pre_trigger_score'] = row['density_score']*0.6 + count*0.4

zones_15.head()

## 8. Simulação de estratégia e métricas de win/loss, reward/risk

Backtest simples: entrada em rompimento/rejeição de zona de gatilho (15m), SL=sl_minimo, TP=sg_minimo, horizonte 10min.

In [ ]:
print('Iniciando simulação de trades 15min (Top 200 by pre_trigger_score)...')
trades = []
for idx,row in zones_15.sort_values('pre_trigger_score', ascending=False).head(200).iterrows():
    ctime = row['datetime']
    prev = data['5min'][data['5min'].datetime <= ctime].tail(1)
    if prev.empty:
        continue
    entry = float(prev['Fechamento'].iloc[-1])
    fval = float(prev['fForca'].iloc[-1])
    if abs(fval) < 55:
        continue
    direction = 'buy' if fval > 0 else 'sell'
    target = entry + sg_minimo if direction == 'buy' else entry - sg_minimo
    stop = entry - sl_minimo if direction == 'buy' else entry + sl_minimo
    future = data['5min'][(data['5min'].datetime > ctime) & (data['5min'].datetime <= ctime + pd.Timedelta('10m'))]
    if future.empty:
        continue
    outcome = 'neutral'
    best_take = 0.0
    worst_stop = float('inf')
    for _,frow in future.iterrows():
        if direction == 'buy':
            if frow['Mínimo'] <= stop:
                outcome='loss'; break
            if frow['Máximo'] >= target:
                outcome='win'; break
            walk_profit = frow['Máximo'] - entry
            walk_loss = entry - frow['Mínimo']
        else:
            if frow['Máximo'] >= stop:
                outcome='loss'; break
            if frow['Mínimo'] <= target:
                outcome='win'; break
            walk_profit = entry - frow['Mínimo']
            walk_loss = frow['Máximo'] - entry
        if walk_profit > best_take:
            best_take = walk_profit
        if walk_loss < worst_stop:
            worst_stop = walk_loss
    if outcome == 'neutral':
        last = future.iloc[-1]['Fechamento']
        if direction == 'buy':
            outcome = 'win' if last >= target else 'loss' if last <= stop else 'neutral'
        else:
            outcome = 'win' if last <= target else 'loss' if last >= stop else 'neutral'
    trades.append({
        'datetime': ctime,
        'dir': direction,
        'entry': entry,
        'target': target,
        'stop': stop,
        'fForca': fval,
        'result': outcome,
        'pre_trigger_score': row['pre_trigger_score'],
        'best_take': round(best_take, 4),
        'worst_stop': round(worst_stop if worst_stop != float('inf') else sl_minimo, 4)
    })

trades_df = pd.DataFrame(trades)
print('Número de trades simulados:', len(trades_df))
if not trades_df.empty:
    print('Trade samples:')
    print(trades_df.head(10))
    wins = (trades_df['result'] == 'win').sum()
    losses = (trades_df['result'] == 'loss').sum()
    neutrals = (trades_df['result'] == 'neutral').sum()
    winrate = wins / (wins + losses) if (wins + losses) > 0 else 0
    avg_rr = (sg_minimo / sl_minimo) if sl_minimo > 0 else np.nan
    max_profit = trades_df['best_take'].max()
    min_stop = trades_df['worst_stop'].min()
    print('Trades', len(trades_df), 'Wins', wins, 'Losses', losses, 'Neutrals', neutrals, 'Winrate', round(winrate, 4), 'Avg RR', round(avg_rr, 2))
    print('Maior range de lucro potencial:', round(max_profit, 4), 'Menor stop loss observado:', round(min_stop, 4))
else:

    print('Nenhuma trade gerada')

trades_df.head()


## 9. Relatório de resultados com JSON de áreas e métricas de win

Exportar áreas + zonas + métricas (win_rate, avg_reward_risk, trades) em JSON e sumarizar melhor abordagem.

In [ ]:
import json

report = {
    'settings': {'x_pontos': x_pontos, 'sl_minimo': sl_minimo, 'sg_minimo': sg_minimo},
    'zones_15': zones_15.head(20).to_dict('records'),
    'trades': trades_df.to_dict('records'),
}

# Converte datetime para serializável em JSON
for item in report['zones_15']:
    if 'datetime' in item and not isinstance(item['datetime'], str):
        item['datetime'] = str(item['datetime'])
for t in report['trades']:
    if 'datetime' in t and not isinstance(t['datetime'], str):
        t['datetime'] = str(t['datetime'])

with open('e:/repo/PriceAction_Fisica/WESQUAD/DadosCandlesBacktest/2024_26/analysis_results_2024_26.json','w',encoding='utf-8') as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

report